# Lab 6 · Dữ liệu ngoài: file lớn, API lịch sử & DuckDB

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · Bài 6**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook demo bài 6 gọi API dự báo thời tiết. Lab này đọc có chọn lọc **bảng đầy đủ 90 cột**,
lấy **thời tiết quá khứ** rồi ghép với số review theo ngày, đồng thời dùng DuckDB xử lý
690.112 dòng review và đối chiếu với kết quả ở lab 2.

**Mục tiêu bài lab**

1. Đọc file lớn có chọn lọc bằng `usecols` + `parse_dates` và đo chênh lệch thời gian đọc.
2. Làm sạch cột giá dạng chuỗi; ghi Parquet và so kích thước/kiểu dữ liệu với CSV.
3. Lưu response thô của API lịch sử (archive), đếm review theo ngày và ghép hai nguồn.
4. Viết truy vấn DuckDB có `GROUP BY`, `JOIN` — và kiểm chứng chéo bằng con số đã biết.

Phiên bản đề: `external-data-duckdb-v1` (25/09/2026). Lưu đúng tên **`lab-06.ipynb`**.

## Cách làm và nộp bài

- Phần khởi động và các bài có hướng dẫn: nên **tự gõ, không dùng AI** — các bài kiểm tra
  định kỳ 🚫 ở giờ lý thuyết đánh giá các kỹ năng này. Bài tự làm cuối lab được gắn nhãn ✅ mở:
  được dùng AI, kèm trách nhiệm khai báo theo chính sách AI của môn.
- Ghi dự đoán trước khi chạy; điền thân hàm TODO và phần trả lời Markdown.
- Giữ tên hàm, tham số, cell ID và metadata; viết helper/import thêm trong chính cell bài làm
  (Bài 7–8 đã có sẵn `import duckdb` trong thân hàm). Grader chỉ chạy cell câu đó với file/input riêng,
  không phụ thuộc biến `rv`, `nho`, `DATA_DIR`… ở cell khác. `pd` và `np` được cung cấp sẵn khi chấm.
- Grader **không có mạng**: hàm nhận đường dẫn file hoặc dict response làm tham số, không tự tải URL hay gọi API.
- Public checks chỉ là ví dụ tự kiểm tra trên dữ liệu giả lập nhỏ; test ẩn dùng file khác cùng hợp đồng.
  Một check lỗi không dừng các check sau. TODO báo **CHƯA LÀM**; lỗi cú pháp vẫn cần sửa.
- Không sửa DataFrame/Series/dict đầu vào; không ghi cứng kết quả. Không cần xử lý input ngoài hợp đồng.
- Lab chạy với hành vi **Copy-on-Write** của pandas 3 (bật sẵn trong cell dữ liệu và khi chấm).
- Nếu chưa giải quyết được một bước sau 3 phút, hãy gọi giảng viên thực hành đến hỗ trợ.
- **Restart & Run all**, lưu notebook, commit/push vào repo của bạn trong org và tạo tag theo cuối bài.
  Không commit các file dữ liệu do notebook tạo ra (`data/`, `data_demo/`, `raw/`, `t6.*`).

## Rubric dự kiến — 100 điểm

| Mục | Điểm | Cách đánh giá |
|---|---:|---|
| Bài 1–8, mỗi bài 10 điểm | 80 | Tự động; giảng viên kiểm tra phương pháp |
| Diễn giải: thời gian đọc và Parquet, tương quan, kết quả JOIN | 15 | Giảng viên |
| Khai báo AI và bằng chứng kiểm chứng | 5 | Giảng viên |

Yêu cầu dùng `usecols`, `str.replace(..., regex=False)`, `to_parquet`, `merge`, SQL trong DuckDB được giảng viên
kiểm tra cùng code, không suy ra chỉ từ kết quả. Bài tự làm ✅ mở không cộng vào 80 điểm tự động.

In [ ]:
# Public checks giúp tự kiểm tra; đây không phải điểm chính thức.
# Không sửa cell này trong bài nộp.
PUBLIC_RESULTS = {}

def public_check(case_id, check):
    try:
        check()
    except NotImplementedError:
        status, detail = "CHƯA LÀM", "Điền phần TODO rồi chạy lại."
    except AssertionError as exc:
        status, detail = "CHƯA ĐẠT", str(exc) or "Kết quả chưa khớp ví dụ công khai."
    except Exception as exc:
        status, detail = "LỖI", f"{type(exc).__name__}: {exc}"
    else:
        status, detail = "ĐẠT", "Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric."
    PUBLIC_RESULTS[case_id] = status
    print(f"[{status}] {case_id}: {detail}")

def require_answer(value):
    if value is None or value is Ellipsis:
        raise NotImplementedError

def preview(label, action):
    try:
        value = action()
    except NotImplementedError:
        print(f"{label}: chưa chạy được vì còn TODO.")
    except Exception as exc:
        print(f"{label}: {type(exc).__name__}: {exc}")
    else:
        print(label)
        print(value)

def show_public_summary():
    print("PUBLIC CHECKS — không phải điểm chính thức")
    for case_id, status in PUBLIC_RESULTS.items():
        print(f"{case_id}: {status}")
    print("Sau khi sửa, Restart & Run all để làm mới toàn bộ kết quả.")

## Dữ liệu và thư viện

Lab cần `duckdb` và `pyarrow` (Colab có sẵn). Nếu cell dưới báo thiếu, chạy `%pip install duckdb pyarrow`
trong một cell riêng rồi Restart.

Mặc định `USE_SNAPSHOT = False`: notebook ghi **dữ liệu giả lập nhỏ** (51 review, 6 listing, response thời tiết
10 ngày) vào thư mục `data_demo/` để Run all được cả khi không có mạng. Không dùng kết quả giả lập để kết luận về Santiago.

Đặt `USE_SNAPSHOT = True` để tải ba file của snapshot Santiago **2026-06-29** từ Inside Airbnb vào `data/`
(`visualisations/reviews.csv`, `visualisations/listings.csv`, `data/listings.csv.gz` — bảng đầy đủ 90 cột)
và gọi API archive của Open-Meteo cho tháng 1/2025. Public checks luôn dùng dữ liệu giả lập.

In [ ]:
try:
    import duckdb
    import pyarrow
    print("duckdb", duckdb.__version__, "· pyarrow", pyarrow.__version__)
except ImportError as exc:
    print(f"Thiếu thư viện {exc.name}: chạy %pip install duckdb pyarrow trong một cell riêng rồi Restart.")

In [ ]:
import json
import os
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

# Hành vi pandas 3 (Copy-on-Write).
if pd.__version__.startswith("2."):
    pd.set_option("mode.copy_on_write", True)

SNAPSHOT_BASE = "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/"
URL_RV = SNAPSHOT_BASE + "visualisations/reviews.csv"
URL_LISTINGS = SNAPSHOT_BASE + "visualisations/listings.csv"
URL_FULL = SNAPSHOT_BASE + "data/listings.csv.gz"

# Dữ liệu GIẢ LẬP nhỏ, cùng cấu trúc với các file Inside Airbnb và response Open-Meteo.
_ngay = []
for ngay, so in zip(pd.date_range("2025-01-01", "2025-01-10"), [3, 5, 2, 6, 0, 4, 7, 1, 5, 3]):
    _ngay += [ngay] * so
_ngay += list(pd.to_datetime(["2023-03-15", "2023-08-02", "2023-11-20", "2024-02-11", "2024-05-05",
                              "2024-07-19", "2024-09-30", "2024-12-24", "2024-12-31", "2025-03-08",
                              "2025-06-21", "2025-09-18", "2025-12-05", "2026-02-14", "2026-04-01"]))
DEMO_REVIEWS = pd.DataFrame({
    "listing_id": [[11, 12, 11, 13, 14, 11, 15, 12, 99][k % 9] for k in range(len(_ngay))],
    "date": _ngay,
})
DEMO_LISTINGS_FULL = pd.DataFrame({
    "id": [11, 12, 13, 14, 15, 16],
    "name": ["Depto Lastarria", "Pieza Bellavista", "Loft Providencia", "Casa Ñuñoa",
             "Studio Centro", "Depto nuevo"],
    "host_id": [501, 502, 501, 503, 504, 505],
    "neighbourhood_cleansed": ["Santiago", "Recoleta", "Providencia", "Ñuñoa", "Santiago", "Las Condes"],
    "room_type": ["Entire home/apt", "Private room", "Entire home/apt", "Entire home/apt",
                  "Entire home/apt", "Private room"],
    "price": ["$45,647.00", "$18,500.00", "$1,120,000.00", None, "$59,000.00", "$25,990.50"],
    "first_review": ["2023-03-15", "2023-08-02", "2024-02-11", "2023-11-20", "2025-01-02", None],
    "number_of_reviews": [14, 9, 8, 4, 6, 0],
    "review_scores_rating": [4.91, 4.75, 4.88, 5.0, 4.6, None],
})
DEMO_WEATHER = {
    "latitude": -33.45, "longitude": -70.66, "timezone": "America/Santiago",
    "daily_units": {"time": "iso8601", "temperature_2m_max": "°C"},
    "daily": {"time": [f"2025-01-{d:02d}" for d in range(1, 11)],
              "temperature_2m_max": [31.4, 33.1, 29.1, 32.8, 30.2, 30.0, 28.5, 31.8, 30.9, 27.6]},
}

# Đặt USE_SNAPSHOT = True để tải 3 file Santiago và gọi API archive thật.
USE_SNAPSHOT = False

if USE_SNAPSHOT:
    DATA_DIR = Path("data")
    DATA_DIR.mkdir(exist_ok=True)
    for ten, url in [("reviews.csv", URL_RV), ("listings.csv", URL_LISTINGS),
                     ("listings_full.csv.gz", URL_FULL)]:
        if not (DATA_DIR / ten).exists():
            urllib.request.urlretrieve(url, DATA_DIR / ten)
    FULL_PATH = DATA_DIR / "listings_full.csv.gz"
    KY = ("2025-01-01", "2025-01-31")
    DATA_LABEL = "Snapshot Santiago 2026-06-29"
else:
    DATA_DIR = Path("data_demo")
    DATA_DIR.mkdir(exist_ok=True)
    DEMO_REVIEWS.to_csv(DATA_DIR / "reviews.csv", index=False)
    DEMO_LISTINGS_FULL[["id", "name", "room_type"]].to_csv(DATA_DIR / "listings.csv", index=False)
    DEMO_LISTINGS_FULL.to_csv(DATA_DIR / "listings_full.csv", index=False)
    FULL_PATH = DATA_DIR / "listings_full.csv"
    KY = ("2025-01-01", "2025-01-10")
    DATA_LABEL = "GIẢ LẬP — 51 review, 6 listing, 10 ngày thời tiết"

REVIEWS_PATH = DATA_DIR / "reviews.csv"
LISTINGS_PATH = DATA_DIR / "listings.csv"
print(DATA_LABEL, "·", REVIEWS_PATH, LISTINGS_PATH, FULL_PATH)

## Bài 1 · Đọc có chọn lọc: `usecols` + `parse_dates` (khởi động + bảng 90 cột)

Ở các bài trước, dữ liệu được đọc từ bản rút gọn có 19 cột. Bảng **đầy đủ** từ Inside Airbnb có 90 cột
và lưu giá dưới dạng chuỗi `"$45,647.00"`. Chỉ đọc đúng cột cần, và parse cột ngày **ngay khi đọc**.

### Hợp đồng hàm · 10 điểm tự động

`read_selected(path, columns: list, date_columns: list) -> pd.DataFrame`

Nhận đường dẫn tới file CSV (có thể nén `.gz`), danh sách cột cần đọc (đều có trong file) và danh sách cột
ngày (con của `columns`, có thể rỗng). Đọc bằng `pd.read_csv(..., usecols=..., parse_dates=...)`.
Trả DataFrame có **đúng** các cột trong `columns` (thứ tự theo file hay theo danh sách đều được), giữ mọi dòng;
các cột trong `date_columns` có dtype `datetime64`, ô ngày trống thành `NaT`.
Không đọc cả bảng rồi mới chọn cột. Đối chiếu snapshot: `reviews.csv` 690.112 dòng; bảng đầy đủ `(18534, 90)`,
đọc 5 cột được `(18534, 5)`.

In [ ]:
def read_selected(path, columns: list, date_columns: list) -> pd.DataFrame:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q1():
    import tempfile
    cot = ["id", "room_type", "price", "first_review", "number_of_reviews"]
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / "listings_full.csv"
        DEMO_LISTINGS_FULL.to_csv(path, index=False)
        nho = read_selected(path, cot, ["first_review"])
        rv_path = Path(directory) / "reviews.csv"
        DEMO_REVIEWS.to_csv(rv_path, index=False)
        rv_demo = read_selected(rv_path, ["listing_id", "date"], ["date"])
    assert isinstance(nho, pd.DataFrame) and nho.shape == (6, 5)
    assert set(nho.columns) == set(cot)
    assert str(nho["first_review"].dtype).startswith("datetime64"), "Cần parse_dates cho first_review"
    assert nho["first_review"].isna().sum() == 1
    assert len(rv_demo) == 51 and str(rv_demo["date"].dtype).startswith("datetime64")

public_check("Q1 · ví dụ công khai", check_q1)

In [ ]:
def thuc_hanh_q1():
    global rv, nho
    t0 = time.perf_counter()
    full = pd.read_csv(FULL_PATH)                     # Cách 1: đọc cả bảng — cho sẵn để so sánh
    t_full = time.perf_counter() - t0
    t0 = time.perf_counter()
    nho = read_selected(FULL_PATH, ["id", "room_type", "price", "first_review", "number_of_reviews"],
                        ["first_review"])            # Cách 2: chỉ đọc 5 cột
    t_nho = time.perf_counter() - t0
    rv = read_selected(REVIEWS_PATH, ["listing_id", "date"], ["date"])
    return (f"Cả bảng: {full.shape} — {t_full:.2f}s · 5 cột: {nho.shape} — {t_nho:.2f}s\n"
            f"reviews: {len(rv):,} dòng, date: {rv['date'].dtype}")

preview("Bài 1 trên dữ liệu đang dùng", thuc_hanh_q1)

## Bài 2 · Làm sạch cột giá dạng chuỗi

`"$45,647.00"` → `45647.0`: bỏ ký hiệu `$` và dấu phẩy phân cách hàng nghìn rồi đổi kiểu.
Dự đoán trước: nếu dùng `str.replace("$", "", regex=True)` thì điều gì xảy ra? (Trong regex, `$` nghĩa là *cuối chuỗi*.)

### Hợp đồng hàm · 10 điểm tự động

`clean_price(prices: pd.Series) -> pd.Series`

Nhận Series chuỗi dạng `"$1,234.50"` (luôn có `$`, dấu phẩy nghìn tùy độ lớn) hoặc giá trị thiếu.
Trả Series `float64` cùng index: bỏ `$` và `,` bằng **hai lần** `str.replace(..., regex=False)` rồi `astype(float)`;
giá trị thiếu vẫn là NaN. Không sửa input. Đối chiếu snapshot: trung vị giá sau khi làm sạch là **59.000** CLP.

In [ ]:
def clean_price(prices: pd.Series) -> pd.Series:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q2():
    result = clean_price(DEMO_LISTINGS_FULL["price"])
    assert isinstance(result, pd.Series) and result.dtype == np.float64
    np.testing.assert_allclose(result, [45647.0, 18500.0, 1120000.0, np.nan, 59000.0, 25990.5])
    assert result.median() == 45647.0
    assert DEMO_LISTINGS_FULL["price"].iloc[0] == "$45,647.00", "Không sửa Series đầu vào"

public_check("Q2 · ví dụ công khai", check_q2)

In [ ]:
def thuc_hanh_q2():
    global nho
    nho = nho.assign(price_num=clean_price(nho["price"]))
    return f"Trung vị giá: {nho['price_num'].median():,.0f} CLP · thiếu {nho['price_num'].isna().sum()} giá"

preview("Bài 2 trên dữ liệu đang dùng", thuc_hanh_q2)

## Bài 3 · CSV hay Parquet: kích thước và kiểu dữ liệu

### Hợp đồng hàm · 10 điểm tự động

`save_formats(df: pd.DataFrame, folder) -> dict`

Nhận DataFrame (có thể có cột `datetime64`) và thư mục đã tồn tại. Ghi `folder/"t6.csv"` bằng
`to_csv(..., index=False)` và `folder/"t6.parquet"` bằng `to_parquet(..., index=False)` (ghi đè nếu có).
Đọc lại hai file (`pd.read_csv` **không** `parse_dates`, `pd.read_parquet`). Trả dict có đúng các khóa:

- `size_csv`, `size_parquet`: kích thước hai file theo byte (`os.path.getsize`), kiểu int;
- `dtypes_csv`, `dtypes_parquet`: dict `{tên cột: str(dtype)}` của hai bảng **đọc lại**.

Không sửa input. Đối chiếu snapshot (5 cột + `price_num`): Parquet nhỏ hơn CSV khoảng 3 lần và giữ `datetime64`.

In [ ]:
def save_formats(df: pd.DataFrame, folder) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q3():
    import tempfile
    bang = DEMO_LISTINGS_FULL[["id", "room_type"]].assign(
        first_review=pd.to_datetime(DEMO_LISTINGS_FULL["first_review"]))
    with tempfile.TemporaryDirectory() as directory:
        result = save_formats(bang, Path(directory))
        size_csv = os.path.getsize(Path(directory) / "t6.csv")
        size_pq = os.path.getsize(Path(directory) / "t6.parquet")
    assert set(result) == {"size_csv", "size_parquet", "dtypes_csv", "dtypes_parquet"}
    assert result["size_csv"] == size_csv and result["size_parquet"] == size_pq
    assert result["dtypes_csv"]["first_review"] == "object", "CSV không lưu kiểu ngày"
    assert result["dtypes_parquet"]["first_review"].startswith("datetime64"), "Parquet giữ kiểu ngày"
    assert result["dtypes_parquet"]["id"] == "int64"

public_check("Q3 · ví dụ công khai", check_q3)

In [ ]:
def thuc_hanh_q3():
    kq = save_formats(nho, Path("."))
    return (f"CSV {kq['size_csv']:,} byte — Parquet {kq['size_parquet']:,} byte\n"
            f"first_review: CSV {kq['dtypes_csv']['first_review']} · Parquet {kq['dtypes_parquet']['first_review']}")

preview("Bài 3 trên dữ liệu đang dùng", thuc_hanh_q3)

**Nhận xét** (ghi rõ dữ liệu giả lập hay Santiago): đọc 5 cột nhanh hơn cả bảng bao nhiêu lần? File nào nhỏ hơn,
vì sao? Kiểu của `first_review` sau khi đọc lại mỗi file là gì và điều đó ảnh hưởng gì đến bước sau? …

## Bài 4 · API lịch sử: lưu response thô rồi mới chuyển thành bảng

Câu hỏi: *trong tháng 1/2025, ngày có nhiệt độ cao hơn có nhiều review hơn không?* Cần hai nguồn: thời tiết
**quá khứ** từ API archive của Open-Meteo và số review theo ngày. Khoảng thời gian cố định giúp quy trình dễ tái lập;
lưu **response thô** (kèm tên file có mốc thời gian) để chạy lại không cần gọi API.

### Hợp đồng hàm · 10 điểm tự động

`weather_table(payload: dict, path) -> pd.DataFrame`

Nhận dict response của API (có khóa `"daily"` là dict các list cùng độ dài, luôn có `"time"`) và đường dẫn file
(thư mục cha đã tồn tại). Ghi **toàn bộ** `payload` ra `path` bằng `json.dump` (UTF-8, `ensure_ascii=False`),
rồi trả `pd.DataFrame(payload["daily"])` — đủ mọi biến trong `daily`, giữ thứ tự cột, cột `time` vẫn là chuỗi.
Không sửa `payload`. Đối chiếu snapshot: tháng 1/2025 có 31 ngày, nhiệt độ cao nhất trung bình ~30–31°C.

In [ ]:
def weather_table(payload: dict, path) -> pd.DataFrame:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q4():
    import tempfile
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / "weather_scl_2025-01.json"
        result = weather_table(DEMO_WEATHER, path)
        assert path.exists(), "Cần lưu response thô"
        assert json.loads(path.read_text(encoding="utf-8")) == DEMO_WEATHER
    assert isinstance(result, pd.DataFrame) and result.shape == (10, 2)
    assert list(result.columns) == ["time", "temperature_2m_max"]
    assert result.loc[0, "time"] == "2025-01-01" and round(result["temperature_2m_max"].mean(), 2) == 30.54

public_check("Q4 · ví dụ công khai", check_q4)

In [ ]:
def lay_thoi_tiet():
    global thoi_tiet
    if USE_SNAPSHOT:
        import requests
        r = requests.get(
            "https://archive-api.open-meteo.com/v1/archive",
            params={"latitude": -33.45, "longitude": -70.66,
                    "start_date": KY[0], "end_date": KY[1],
                    "daily": "temperature_2m_max", "timezone": "auto"},
            timeout=30,
        )
        r.raise_for_status()
        payload = r.json()
    else:
        payload = DEMO_WEATHER
    Path("raw").mkdir(exist_ok=True)
    thoi_tiet = weather_table(payload, Path("raw") / f"weather_scl_{KY[0][:7]}.json")
    return (f"{len(thoi_tiet)} ngày, nhiệt độ cao nhất trung bình {thoi_tiet['temperature_2m_max'].mean():.1f}°C\n"
            f"{thoi_tiet.head(3)}")

preview("Bài 4 — thời tiết của kỳ đang dùng", lay_thoi_tiet)

## Bài 5 · Đếm review theo ngày trong một khoảng

### Hợp đồng hàm · 10 điểm tự động

`reviews_per_day(rv: pd.DataFrame, start: str, end: str) -> pd.DataFrame`

Nhận bảng review có cột `date` kiểu `datetime64` (chỉ ngày, không giờ) và hai mốc `"YYYY-MM-DD"`.
Lọc các dòng có `start <= date <= end` (**gồm cả hai đầu**), đếm số dòng theo ngày
(`groupby(rv["date"].dt.strftime("%Y-%m-%d")).size()`) và trả DataFrame **đúng hai cột** `time` (chuỗi
`"YYYY-MM-DD"`, để ghép với `time` của API) và `so_review` (int), sắp `time` tăng dần, index `0, 1, …`.
Chỉ gồm các ngày có ít nhất một review. Không sửa input.
Đối chiếu snapshot: tháng 1/2025 có **14.636** review, đủ 31 ngày.

In [ ]:
def reviews_per_day(rv: pd.DataFrame, start: str, end: str) -> pd.DataFrame:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q5():
    result = reviews_per_day(DEMO_REVIEWS, "2025-01-01", "2025-01-10")
    assert isinstance(result, pd.DataFrame) and list(result.columns) == ["time", "so_review"]
    assert len(result) == 9, "Ngày 05/01 không có review nên không xuất hiện"
    assert result["so_review"].sum() == 36
    assert result.loc[0, "time"] == "2025-01-01" and result.loc[0, "so_review"] == 3
    assert result["time"].tolist() == sorted(result["time"])
    assert reviews_per_day(DEMO_REVIEWS, "2025-01-10", "2025-01-10")["so_review"].tolist() == [3]

public_check("Q5 · ví dụ công khai", check_q5)

## Bài 6 · Ghép thời tiết với số review theo ngày

### Hợp đồng hàm · 10 điểm tự động

`join_weather(weather: pd.DataFrame, per_day: pd.DataFrame) -> dict`

Nhận bảng thời tiết (cột `time` chuỗi, `temperature_2m_max`) và bảng của Bài 5 (`time`, `so_review`).
`merge` theo `time` với `how="left"` (giữ mọi ngày của thời tiết, thứ tự của `weather`). Trả dict có đúng các khóa:

- `gop`: DataFrame sau merge (cột của `weather` rồi `so_review`); ngày không có review để NaN, **không** tự điền 0;
- `so_thieu`: int, số NaN ở `so_review`;
- `he_so`: float, hệ số tương quan `gop["temperature_2m_max"].corr(gop["so_review"])` (pandas bỏ cặp có NaN).

Có ít nhất hai ngày khớp. Không sửa input. Đối chiếu snapshot: `gop` có shape `(31, 3)`, không thiếu, hệ số ≈ **0,08**.

In [ ]:
def join_weather(weather: pd.DataFrame, per_day: pd.DataFrame) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q6():
    weather = pd.DataFrame(DEMO_WEATHER["daily"])
    per_day = pd.DataFrame({"time": [f"2025-01-{d:02d}" for d in (1, 2, 3, 4, 6, 7, 8, 9, 10)],
                            "so_review": [3, 5, 2, 6, 4, 7, 1, 5, 3]})
    result = join_weather(weather, per_day)
    assert set(result) == {"gop", "so_thieu", "he_so"}
    gop = result["gop"]
    assert gop.shape == (10, 3) and list(gop.columns) == ["time", "temperature_2m_max", "so_review"]
    assert result["so_thieu"] == 1 and np.isnan(gop.loc[4, "so_review"])
    assert round(result["he_so"], 3) == 0.084

public_check("Q6 · ví dụ công khai", check_q6)

In [ ]:
def thuc_hanh_q5_q6():
    theo_ngay = reviews_per_day(rv, *KY)
    kq = join_weather(thoi_tiet, theo_ngay)
    return (f"{theo_ngay['so_review'].sum():,} review trong {len(theo_ngay)} ngày có review · "
            f"{kq['so_thieu']} ngày không có review\nHệ số tương quan: {kq['he_so']:.3f}")

preview("Bài 5–6 trên dữ liệu đang dùng", thuc_hanh_q5_q6)

**Diễn giải:** hệ số tương quan … cho thấy … Ngày không có review nên để NaN hay điền 0? Khi diễn giải cần lưu ý gì? …

## Bài 7 · DuckDB: truy vấn trực tiếp trên file CSV

DuckDB truy vấn file mà không cần tải toàn bộ vào DataFrame: `FROM 'đường/dẫn/reviews.csv'`.

### Hợp đồng hàm · 10 điểm tự động

`sql_reviews_by_year(reviews_path) -> pd.DataFrame`

Nhận đường dẫn file CSV review (cột `listing_id`, `date`). Chạy **một truy vấn DuckDB** trên file
(`SELECT year(date) AS nam, count(*) AS n FROM '<path>' GROUP BY nam ORDER BY nam`) và trả `.df()`:
DataFrame đúng hai cột `nam`, `n` (số nguyên), `nam` tăng dần. Đổi `reviews_path` sang chuỗi bằng `str(...)`.
Đối chiếu chéo: năm 2025 của snapshot có **211.432** review — phải khớp con số đếm bằng dict ở lab 2.

In [ ]:
def sql_reviews_by_year(reviews_path) -> pd.DataFrame:
    import duckdb
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q7():
    import tempfile
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / "reviews.csv"
        DEMO_REVIEWS.to_csv(path, index=False)
        result = sql_reviews_by_year(path)
    assert isinstance(result, pd.DataFrame) and list(result.columns) == ["nam", "n"]
    assert result["nam"].tolist() == [2023, 2024, 2025, 2026]
    assert result["n"].tolist() == [3, 6, 40, 2]
    doi_chieu = DEMO_REVIEWS["date"].dt.year.value_counts().sort_index()
    assert result["n"].tolist() == doi_chieu.tolist(), "SQL và pandas phải cho cùng đáp số"

public_check("Q7 · ví dụ công khai", check_q7)

## Bài 8 · DuckDB `JOIN`: review theo loại phòng

### Hợp đồng hàm · 10 điểm tự động

`sql_reviews_by_room_type(reviews_path, listings_path, nam: int) -> pd.DataFrame`

Nhận đường dẫn file review (`listing_id`, `date`), file listing (có `id`, `room_type`) và một năm.
Một truy vấn DuckDB: `FROM '<reviews>' r JOIN '<listings>' l ON r.listing_id = l.id`,
`WHERE year(r.date) = nam`, `GROUP BY room_type`, trả DataFrame đúng hai cột `room_type`, `n`
sắp `n` **giảm dần**, đồng hạng thì `room_type` tăng dần. Review của listing không có trong file listing bị loại (JOIN trong).
Đối chiếu snapshot: năm 2025 nhóm nguyên căn có **192.326** review, đứng đầu.

In [ ]:
def sql_reviews_by_room_type(reviews_path, listings_path, nam: int) -> pd.DataFrame:
    import duckdb
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q8():
    import tempfile
    with tempfile.TemporaryDirectory() as directory:
        rv_path, ls_path = Path(directory) / "reviews.csv", Path(directory) / "listings.csv"
        DEMO_REVIEWS.to_csv(rv_path, index=False)
        DEMO_LISTINGS_FULL[["id", "name", "room_type"]].to_csv(ls_path, index=False)
        result = sql_reviews_by_room_type(rv_path, ls_path, 2025)
    assert isinstance(result, pd.DataFrame) and list(result.columns) == ["room_type", "n"]
    assert result["room_type"].tolist() == ["Entire home/apt", "Private room"]
    assert result["n"].tolist() == [27, 9]
    assert result["n"].sum() == 36, "4 review năm 2025 của listing 99 không có trong file listing"

public_check("Q8 · ví dụ công khai", check_q8)

In [ ]:
def thuc_hanh_sql():
    dem_nam = sql_reviews_by_year(REVIEWS_PATH)
    theo_loai = sql_reviews_by_room_type(REVIEWS_PATH, LISTINGS_PATH, 2025)
    tong_2025 = int(dem_nam.loc[dem_nam["nam"] == 2025, "n"].iloc[0])
    return (f"{dem_nam}\n\nNăm 2025: {tong_2025:,} review; sau JOIN còn {theo_loai['n'].sum():,}\n{theo_loai}")

preview("Bài 7–8 trên dữ liệu đang dùng", thuc_hanh_sql)

**Diễn giải JOIN:** tổng `n` sau JOIN so với số review cả năm ở Bài 7 thế nào, vì sao? Tỷ trọng review của nhóm
nguyên căn so với tỷ trọng listing của nhóm này nói lên điều gì? …

## Bài tự làm ✅ mở (làm sớm tại lớp hoặc làm tại nhà) — không cộng vào 80 điểm tự động

Bạn được dùng AI theo quy trình 5 bước; hãy ghi lại prompt và cách kiểm chứng.

### Tự làm 1 · So sánh với tháng 7

Lặp Bài 4–6 cho **tháng 7/2025** (giữa đông): gọi archive API, đếm review theo ngày, merge,
tính tương quan. So với tháng 1 và viết 2 câu nhận xét. Lưu response thô vào `raw/`.

### Tự làm 2 · Truy vấn listing có nhiều review

Viết truy vấn DuckDB tìm **5 listing nhiều review nhất năm 2025** kèm tên (JOIN lấy `name`).
Kiểm chứng con số đầu bảng bằng pandas (`rv` + lọc năm + `value_counts` trên `listing_id`).

In [ ]:
# Viết bài tự làm của bạn ở đây (không được chấm tự động).

**Bài tự làm của tôi:** …

## Khai báo sử dụng AI

- **Công cụ đã dùng:** … *(hoặc “không dùng”)*
- **Bài đã dùng AI và yêu cầu chính:** …
- **Một gợi ý đã kiểm chứng, cách kiểm chứng và kết quả:** …

Không chỉ ghi “code chạy được”; nêu phép tính tay, số dòng hoặc ví dụ đối chiếu đã dùng.

---

## Tóm tắt bài lab

| Nội dung chính | Sẽ gặp lại ở |
|---|---|
| `usecols`/`parse_dates` trên bảng 90 cột | đọc file có nhiều cột một cách có chọn lọc |
| Làm sạch giá chuỗi với `regex=False` | chuẩn hoá cột số lưu dạng chữ |
| Parquet lưu dtype, nhỏ hơn khi dữ liệu lớn | dữ liệu trung gian trong `processed/` |
| API lịch sử và lưu response thô | pipeline sử dụng API ở bài 11 |
| Ghép hai nguồn theo ngày và diễn giải tương quan yếu | phân tích và thẩm định kết luận ở bài 14 |
| DuckDB `GROUP BY`/`JOIN` và đối chiếu với lab 2 | kiểm chứng chéo bằng hai công cụ |

**Trước khi nộp:** hoàn thành TODO và câu trả lời, chạy Restart & Run all, lưu notebook.

**Bài giảng tiếp theo:** xử lý **chuỗi và biểu thức chính quy (regex)**.

## Thông tin và nộp bài

**Họ tên / MSSV / lớp:** [Điền]

1. Giữ tên file **`lab-06.ipynb`** ở gốc repo (dấu gạch ngang, không phải `lab_06.ipynb`).
2. Hoàn thành hàm TODO, các nhận xét, diễn giải và khai báo AI.
3. Restart & Run all; đọc bảng public checks, lưu file rồi commit/push lên `main`.
   Không commit `data/`, `data_demo/`, `raw/`, `t6.csv`, `t6.parquet`.
4. Tạo tag `submit/lab-06/v1`; lần sửa tiếp dùng v2 hoặc v3, không di chuyển tag đã nộp.
5. Repo cần được giảng viên đăng ký chấm. Xem status `private-grader/lab-06` trên commit nộp;
   điểm tự động tối đa 80, còn 20 điểm chờ giảng viên. Xem lịch quét trong README.

Tag v1/v2/v3 là **lần nộp**, không phải phiên bản đề. Không đổi metadata để chọn bộ test khác.

In [ ]:
show_public_summary()